## 1. Setup

In [1]:
import random
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEEDS = [42, 123, 456, 789, 2024]
print(f"Seeds: {SEEDS}")

Seeds: [42, 123, 456, 789, 2024]


In [2]:
EMOTIONS = ['anxiety', 'burnout', 'sadness', 'overwhelm', 'neutral']

GAMES_EXTENDED = [
    "breathing_game", "color_tap", "body_scan",          # anxiety (0-2)
    "nature_walk", "memory_match", "gratitude_journal",  # burnout (3-5)
    "puzzle", "word_search", "trivia",                   # neutral (6-8)
    "bubble_pop", "doodle_draw", "countdown_timer",      # overwhelm (9-11)
    "music_therapy", "micro_journal", "positive_memory"  # sadness (12-14)
]

EMOTION_GAMES = {
    'anxiety':   [0, 1, 2],
    'burnout':   [3, 4, 5],
    'neutral':   [6, 7, 8],
    'overwhelm': [9, 10, 11],
    'sadness':   [12, 13, 14],
}

EMOTION_TO_IDX = {'anxiety': 0, 'burnout': 1, 'sadness': 2, 'overwhelm': 3, 'neutral': 4}

# Population-level reward table (same as in v3/v4 notebooks)
REWARD_TABLE = {
    'anxiety':   [0.90, 0.75, 0.70, 0.30, 0.20, 0.25, 0.20, 0.15, 0.10, 0.15, 0.20, 0.25, 0.20, 0.30, 0.25],
    'burnout':   [0.35, 0.25, 0.30, 0.90, 0.75, 0.80, 0.30, 0.25, 0.20, 0.20, 0.25, 0.30, 0.40, 0.55, 0.45],
    'neutral':   [0.25, 0.30, 0.20, 0.40, 0.45, 0.40, 0.85, 0.80, 0.75, 0.35, 0.40, 0.30, 0.35, 0.40, 0.35],
    'overwhelm': [0.50, 0.45, 0.40, 0.35, 0.25, 0.30, 0.30, 0.25, 0.20, 0.85, 0.80, 0.75, 0.25, 0.30, 0.20],
    'sadness':   [0.25, 0.20, 0.25, 0.45, 0.35, 0.40, 0.30, 0.25, 0.20, 0.25, 0.30, 0.20, 0.60, 0.75, 0.85],
}

N_ARMS = 15
STATE_DIM = 6  # 5 emotion one-hot + stress

STRESS_RANGES = {
    'anxiety':   (0.7, 1.0),
    'burnout':   (0.6, 0.9),
    'sadness':   (0.5, 0.9),
    'overwhelm': (0.7, 1.0),
    'neutral':   (0.0, 0.4),
}

## 3. KEY FIX: persistent per-user reward offsets



In [3]:
def make_user_offsets(n_users, offset_std, seed):
    """Persistent per-user reward offset, sampled once per user (not per draw)."""
    rng = np.random.RandomState(seed)
    return {uid: rng.normal(0, offset_std, N_ARMS) for uid in range(n_users)}


def simulate_reward(emotion, game_idx, user_offset=None, noise_std=0.08):
    """
    Reward = population base (+ optional persistent user offset) + per-episode noise.
    Same functional form as v3/v4 notebooks; only the user_offset term is new.
    """
    base = REWARD_TABLE[emotion][game_idx]
    if user_offset is not None:
        base = base + user_offset[game_idx]
    engagement = np.clip(base + np.random.normal(0, noise_std), 0, 1)
    if np.random.random() < engagement:
        playtime = np.random.uniform(5, 15)
    else:
        playtime = np.random.uniform(0, 1)
    if playtime < 0.5:
        return -0.7
    return round(0.3 + 0.7 * min(1.0, playtime / 10), 3)


def make_vector(emotion, stress):
    one_hot = [0.0] * 5
    one_hot[EMOTION_TO_IDX[emotion]] = 1.0
    return np.array(one_hot + [stress])

## 4. LinUCB bandit 


In [4]:
class LinUCBBandit:
    def __init__(self, alpha=0.5, n_arms=N_ARMS, state_dim=STATE_DIM):
        self.alpha = alpha
        self.A = [np.eye(state_dim) for _ in range(n_arms)]
        self.b = [np.zeros(state_dim) for _ in range(n_arms)]

    def select(self, x, valid_games):
        scores = {}
        for arm in valid_games:
            A_inv = np.linalg.inv(self.A[arm])
            theta = A_inv @ self.b[arm]
            ucb = theta @ x + self.alpha * np.sqrt(x @ A_inv @ x)
            scores[arm] = ucb
        return max(scores, key=scores.get)

    def update(self, arm, x, reward):
        self.A[arm] += np.outer(x, x)
        self.b[arm] += reward * x


def clone_bandit(b):
    nb = LinUCBBandit(alpha=b.alpha)
    nb.A = [a.copy() for a in b.A]
    nb.b = [bb.copy() for bb in b.b]
    return nb


def pretrain_population_bandit(n_episodes=5000, seed=0, alpha=0.5):
    """Population-level pretraining, same as the 10k-episode pretrain step
    used for the main bandit in v3/v4. Used to warm-start both non-personalised
    and personalised policies so neither starts from a cold, uninformative prior."""
    random.seed(seed); np.random.seed(seed)
    bandit = LinUCBBandit(alpha=alpha)
    for _ in range(n_episodes):
        emotion = random.choice(EMOTIONS)
        low, high = STRESS_RANGES[emotion]
        stress = random.uniform(low, high)
        x = make_vector(emotion, stress)
        valid = EMOTION_GAMES[emotion]
        a = random.choice(valid)
        r = simulate_reward(emotion, a, user_offset=None)
        bandit.update(a, x, r)
    return bandit

## 5. Four policies

- **Random** — uniform choice among the 3 valid games for the detected emotion.
- **Population-best (Fixed)** — always the argmax action of the *population* reward
  table per emotion; ignores individual users by construction.
- **Non-personalised LinUCB** — a single LinUCB instance shared across all users
  (sees emotion + stress, never sees user identity).
- **Personalised LinUCB** — a separate LinUCB instance per user, warm-started from
  the same population pretraining, then adapted on that user's own sessions.

In [5]:
def run_single_seed(seed, n_users, n_sessions, offset_std, warm_start=True,
                     pretrain_episodes=5000, alpha=0.5):
    random.seed(seed); np.random.seed(seed)

    user_offsets = make_user_offsets(n_users, offset_std, seed=seed)
    pop_best_action = {
        e: EMOTION_GAMES[e][int(np.argmax([REWARD_TABLE[e][g] for g in EMOTION_GAMES[e]]))]
        for e in EMOTIONS
    }

    pretrained = pretrain_population_bandit(pretrain_episodes, seed=seed, alpha=alpha) if warm_start else None
    non_pers_bandit = clone_bandit(pretrained) if warm_start else LinUCBBandit(alpha=alpha)
    pers_bandits = {
        uid: (clone_bandit(pretrained) if warm_start else LinUCBBandit(alpha=alpha))
        for uid in range(n_users)
    }

    rewards = {'random': [], 'pop_best': [], 'non_pers': [], 'pers': []}

    for uid in range(n_users):
        offset = user_offsets[uid]
        for s in range(n_sessions):
            emotion = random.choice(EMOTIONS)
            low, high = STRESS_RANGES[emotion]
            stress = random.uniform(low, high)
            x = make_vector(emotion, stress)
            valid = EMOTION_GAMES[emotion]

            a = random.choice(valid)
            rewards['random'].append(simulate_reward(emotion, a, offset))

            a = pop_best_action[emotion]
            rewards['pop_best'].append(simulate_reward(emotion, a, offset))

            a = non_pers_bandit.select(x, valid)
            r = simulate_reward(emotion, a, offset)
            non_pers_bandit.update(a, x, r)
            rewards['non_pers'].append(r)

            a = pers_bandits[uid].select(x, valid)
            r = simulate_reward(emotion, a, offset)
            pers_bandits[uid].update(a, x, r)
            rewards['pers'].append(r)

    return {k: float(np.mean(v)) for k, v in rewards.items()}

## 6. Main experiment — matching manuscript's default setup

30 users × 100 sessions, moderate heterogeneity (`offset_std=0.30`), n=5 seeds.
This mirrors the parameters implied by Table 5 of the manuscript. Each seed is one
full simulation; we report mean ± std **across the 5 seed-level means**, and run
independent-samples t-tests on those 5 numbers per comparison — the same protocol
used for Tables 2–4 in the manuscript.

In [6]:
N_USERS = 30
N_SESSIONS = 100
OFFSET_STD_MAIN = 0.30

seed_results = {k: [] for k in ['random', 'pop_best', 'non_pers', 'pers']}
for sd in SEEDS:
    r = run_single_seed(sd, n_users=N_USERS, n_sessions=N_SESSIONS, offset_std=OFFSET_STD_MAIN)
    for k in seed_results:
        seed_results[k].append(r[k])

summary = pd.DataFrame({k: {'mean': np.mean(v), 'std': np.std(v)} for k, v in seed_results.items()}).T
summary['vs_random_pct'] = (summary['mean'] - summary.loc['random', 'mean']) / abs(summary.loc['random', 'mean']) * 100
print(summary.round(4))
print()
for k, v in seed_results.items():
    print(f"{k:10s}: {np.round(v, 4)}")

            mean     std  vs_random_pct
random    0.6204  0.0207         0.0000
pop_best  0.6915  0.0155        11.4551
non_pers  0.6935  0.0200        11.7851
pers      0.6970  0.0172        12.3455

random    : [0.6576 0.5971 0.6249 0.6144 0.6081]
pop_best  : [0.7156 0.6821 0.6916 0.6985 0.6696]
non_pers  : [0.7096 0.6761 0.7164 0.7013 0.6642]
pers      : [0.715  0.6842 0.7026 0.713  0.6703]


In [7]:
def report_ttest(name_a, name_b):
    a, b = np.array(seed_results[name_a]), np.array(seed_results[name_b])
    t, p = stats.ttest_ind(a, b)
    d = (a.mean() - b.mean()) / np.sqrt((a.std(ddof=1)**2 + b.std(ddof=1)**2) / 2)
    print(f"{name_a} vs {name_b}: delta={a.mean()-b.mean():+.4f}  t={t:.3f}  p={p:.4f}  Cohen d={d:.3f}")
    return t, p, d

print("=== Statistical tests (n=5 seed-level means) ===")
report_ttest('pop_best', 'random')
report_ttest('non_pers', 'random')
report_ttest('pers', 'non_pers')
report_ttest('pers', 'pop_best')

=== Statistical tests (n=5 seed-level means) ===
pop_best vs random: delta=+0.0711  t=5.507  p=0.0006  Cohen d=3.483
non_pers vs random: delta=+0.0731  t=5.079  p=0.0010  Cohen d=3.212
pers vs non_pers: delta=+0.0035  t=0.263  p=0.7993  Cohen d=0.166
pers vs pop_best: delta=+0.0055  t=0.477  p=0.6462  Cohen d=0.302


(np.float64(0.4768685583696784),
 np.float64(0.6462097457244795),
 np.float64(0.301598157793827))

## 7. Sensitivity analysis — when *does* personalisation win?




In [8]:
sensitivity_records = []

grid_offset_std = [0.0, 0.15, 0.30, 0.50, 0.80]
grid_n_sessions = [50, 100, 200, 400]

for offset_std in grid_offset_std:
    for n_sessions in grid_n_sessions:
        sr = {k: [] for k in ['non_pers', 'pers']}
        for sd in SEEDS:
            r = run_single_seed(sd, n_users=N_USERS, n_sessions=n_sessions, offset_std=offset_std)
            sr['non_pers'].append(r['non_pers'])
            sr['pers'].append(r['pers'])
        a, b = np.array(sr['pers']), np.array(sr['non_pers'])
        t, p = stats.ttest_ind(a, b)
        sensitivity_records.append({
            'offset_std': offset_std,
            'n_sessions': n_sessions,
            'non_pers_mean': a2 if False else b.mean(),
            'pers_mean': a.mean(),
            'delta': a.mean() - b.mean(),
            't': t,
            'p': p,
            'significant_p<0.05': p < 0.05,
        })

sens_df = pd.DataFrame(sensitivity_records)
pd.set_option('display.width', 120)
print(sens_df.round(4).to_string(index=False))

 offset_std  n_sessions  non_pers_mean  pers_mean   delta       t      p  significant_p<0.05
       0.00          50         0.7641     0.7691  0.0050  0.7228 0.4904               False
       0.00         100         0.7677     0.7676 -0.0002 -0.0460 0.9644               False
       0.00         200         0.7696     0.7693 -0.0003 -0.1608 0.8763               False
       0.00         400         0.7694     0.7668 -0.0026 -1.0836 0.3101               False
       0.15          50         0.7334     0.7510  0.0175  1.7511 0.1180               False
       0.15         100         0.7400     0.7448  0.0048  0.5587 0.5917               False
       0.15         200         0.7444     0.7480  0.0036  0.6748 0.5188               False
       0.15         400         0.7452     0.7471  0.0018  0.2856 0.7825               False
       0.30          50         0.6825     0.6987  0.0162  0.9076 0.3906               False
       0.30         100         0.6935     0.6970  0.0035  0.2629 0.79

In [9]:
pivot = sens_df.pivot(index='offset_std', columns='n_sessions', values='delta')
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto')
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
ax.set_xlabel('Sessions per user')
ax.set_ylabel('Between-user heterogeneity (offset_std)')
ax.set_title('Personalised − Non-personalised reward (Δ)\nGreen = personalisation wins')
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        sig = sens_df[(sens_df.offset_std == pivot.index[i]) & (sens_df.n_sessions == pivot.columns[j])]['significant_p<0.05'].iloc[0]
        ax.text(j, i, f"{val:+.3f}{'*' if sig else ''}", ha='center', va='center', fontsize=9)
plt.colorbar(im, label='Δ reward (personalised − non-personalised)')
plt.tight_layout()
plt.savefig('sensitivity_heatmap.png', dpi=150)
plt.show()
print("'*' marks p < 0.05 (independent t-test, n=5 seed-level means)")

'*' marks p < 0.05 (independent t-test, n=5 seed-level means)


C:\Users\Ciel\AppData\Local\Temp\ipykernel_24404\380582406.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Main results plot — replicating manuscript Table 5 / Figure 5 layout

In [10]:
labels = ['Random', 'Population-best', 'Non-personalised\nLinUCB', 'Personalised\nLinUCB']
keys = ['random', 'pop_best', 'non_pers', 'pers']
means = [np.mean(seed_results[k]) for k in keys]
stds = [np.std(seed_results[k]) for k in keys]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(labels, means, yerr=stds, capsize=5,
              color=['#999999', '#4C72B0', '#55A868', '#C44E52'])
ax.set_ylabel('Mean reward (DM estimate)')
ax.set_title(f'Personalisation hierarchy (mean ± std, n={len(SEEDS)} seeds)\n'
             f'offset_std={OFFSET_STD_MAIN}, {N_USERS} users, {N_SESSIONS} sessions/user')
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, m + 0.02, f'{m:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('personalisation_hierarchy.png', dpi=150)
plt.show()

C:\Users\Ciel\AppData\Local\Temp\ipykernel_24404\2038298955.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Summary table for the manuscript

In [11]:
final_table = pd.DataFrame({
    'Policy': labels,
    'Mean reward': [f'{m:.3f}' for m in means],
    'Std': [f'{s:.3f}' for s in stds],
    'vs Random': [f'{(m/means[0]-1)*100:+.1f}%' if m != means[0] else '—' for m in means],
})
print(final_table.to_string(index=False))

t_np_rand, p_np_rand, _ = report_ttest('non_pers', 'random')
t_pers_np, p_pers_np, d_pers_np = report_ttest('pers', 'non_pers')

print(f"\nFor the manuscript text:")
print(f"Non-pers vs Pop-best: Δ={np.mean(seed_results['non_pers'])-np.mean(seed_results['pop_best']):+.3f}, "
      f"p={stats.ttest_ind(seed_results['non_pers'], seed_results['pop_best'])[1]:.2f}")
print(f"Pers vs Non-pers: Δ={np.mean(seed_results['pers'])-np.mean(seed_results['non_pers']):+.3f}, p={p_pers_np:.2f}, d={d_pers_np:.3f}")

                  Policy Mean reward   Std vs Random
                  Random       0.620 0.021         —
         Population-best       0.691 0.015    +11.5%
Non-personalised\nLinUCB       0.694 0.020    +11.8%
    Personalised\nLinUCB       0.697 0.017    +12.3%
non_pers vs random: delta=+0.0731  t=5.079  p=0.0010  Cohen d=3.212
pers vs non_pers: delta=+0.0035  t=0.263  p=0.7993  Cohen d=0.166

For the manuscript text:
Non-pers vs Pop-best: Δ=+0.002, p=0.88
Pers vs Non-pers: Δ=+0.003, p=0.80, d=0.166


## 10. Statistical power check — per-user analysis (n=30, not n=5)




In [12]:
def run_single_seed_peruser(seed, n_users, n_sessions, offset_std, warm_start=True,
                             pretrain_episodes=5000, alpha=0.5):
    """Same as run_single_seed but returns per-user mean reward (not pooled),
    so we can do a paired test across users instead of across seeds."""
    random.seed(seed); np.random.seed(seed)
    user_offsets = make_user_offsets(n_users, offset_std, seed=seed)
    pretrained = pretrain_population_bandit(pretrain_episodes, seed=seed, alpha=alpha) if warm_start else None
    non_pers_bandit = clone_bandit(pretrained) if warm_start else LinUCBBandit(alpha=alpha)
    pers_bandits = {uid: (clone_bandit(pretrained) if warm_start else LinUCBBandit(alpha=alpha))
                     for uid in range(n_users)}

    per_user_non_pers = {uid: [] for uid in range(n_users)}
    per_user_pers = {uid: [] for uid in range(n_users)}

    for uid in range(n_users):
        offset = user_offsets[uid]
        for s in range(n_sessions):
            emotion = random.choice(EMOTIONS)
            low, high = STRESS_RANGES[emotion]
            stress = random.uniform(low, high)
            x = make_vector(emotion, stress)
            valid = EMOTION_GAMES[emotion]

            a = non_pers_bandit.select(x, valid)
            r = simulate_reward(emotion, a, offset)
            non_pers_bandit.update(a, x, r)
            per_user_non_pers[uid].append(r)

            a = pers_bandits[uid].select(x, valid)
            r = simulate_reward(emotion, a, offset)
            pers_bandits[uid].update(a, x, r)
            per_user_pers[uid].append(r)

    non_pers_means = np.array([np.mean(per_user_non_pers[u]) for u in range(n_users)])
    pers_means = np.array([np.mean(per_user_pers[u]) for u in range(n_users)])
    return non_pers_means, pers_means


# Run at the strongest setting found in the sensitivity grid
non_pers_pu, pers_pu = run_single_seed_peruser(
    seed=42, n_users=30, n_sessions=400, offset_std=0.80
)

t_paired, p_paired = stats.ttest_rel(pers_pu, non_pers_pu)
print(f"Per-user paired t-test (n_users=30, offset_std=0.80, sessions=400):")
print(f"  non_pers mean={non_pers_pu.mean():.4f}  pers mean={pers_pu.mean():.4f}")
print(f"  delta={pers_pu.mean()-non_pers_pu.mean():+.4f}  t={t_paired:.3f}  p={p_paired:.4f}")
print(f"  users where personalised beat non-personalised: {(pers_pu > non_pers_pu).sum()}/{len(pers_pu)}")

Per-user paired t-test (n_users=30, offset_std=0.80, sessions=400):
  non_pers mean=0.6352  pers mean=0.6625
  delta=+0.0273  t=1.405  p=0.1708
  users where personalised beat non-personalised: 17/30


## 11. Root cause found: warm-start prior swamps per-user signal



In [13]:
def run_single_seed_coldstart(seed, n_users, n_sessions, offset_std,
                               alpha_pers=0.5, alpha_nonpers=0.5, pretrain_episodes=5000):
    """Same protocol as run_single_seed, except the personalised bandit
    is cold-started (no shared warm-start prior) instead of cloned from it."""
    random.seed(seed); np.random.seed(seed)
    user_offsets = make_user_offsets(n_users, offset_std, seed=seed)
    pop_best_action = {
        e: EMOTION_GAMES[e][int(np.argmax([REWARD_TABLE[e][g] for g in EMOTION_GAMES[e]]))]
        for e in EMOTIONS
    }

    pretrained = pretrain_population_bandit(pretrain_episodes, seed=seed, alpha=alpha_nonpers)
    non_pers_bandit = clone_bandit(pretrained)                                   # warm-started, as before
    pers_bandits = {uid: LinUCBBandit(alpha=alpha_pers) for uid in range(n_users)}  # COLD START

    rewards = {'random': [], 'pop_best': [], 'non_pers': [], 'pers': []}
    for uid in range(n_users):
        offset = user_offsets[uid]
        for s in range(n_sessions):
            emotion = random.choice(EMOTIONS)
            low, high = STRESS_RANGES[emotion]
            stress = random.uniform(low, high)
            x = make_vector(emotion, stress)
            valid = EMOTION_GAMES[emotion]

            a = random.choice(valid)
            rewards['random'].append(simulate_reward(emotion, a, offset))

            a = pop_best_action[emotion]
            rewards['pop_best'].append(simulate_reward(emotion, a, offset))

            a = non_pers_bandit.select(x, valid)
            r = simulate_reward(emotion, a, offset)
            non_pers_bandit.update(a, x, r)
            rewards['non_pers'].append(r)

            a = pers_bandits[uid].select(x, valid)
            r = simulate_reward(emotion, a, offset)
            pers_bandits[uid].update(a, x, r)
            rewards['pers'].append(r)

    return {k: float(np.mean(v)) for k, v in rewards.items()}


# Same parameters as the main experiment (Section 6) — nothing tuned to force significance
seed_results_fixed = {k: [] for k in ['random', 'pop_best', 'non_pers', 'pers']}
for sd in SEEDS:
    r = run_single_seed_coldstart(sd, n_users=N_USERS, n_sessions=N_SESSIONS, offset_std=OFFSET_STD_MAIN)
    for k in seed_results_fixed:
        seed_results_fixed[k].append(r[k])

summary_fixed = pd.DataFrame({k: {'mean': np.mean(v), 'std': np.std(v)} for k, v in seed_results_fixed.items()}).T
print("Same offset_std / n_users / n_sessions as Section 6, only the warm-start fix applied:")
print(summary_fixed.round(4))

a, b = np.array(seed_results_fixed['pers']), np.array(seed_results_fixed['non_pers'])
t, p = stats.ttest_ind(a, b)
d = (a.mean() - b.mean()) / np.sqrt((a.std(ddof=1)**2 + b.std(ddof=1)**2) / 2)
print(f"\npers vs non_pers (cold-start fix): delta={a.mean()-b.mean():+.4f}  t={t:.3f}  p={p:.4f}  Cohen d={d:.3f}")

Same offset_std / n_users / n_sessions as Section 6, only the warm-start fix applied:
            mean     std
random    0.6204  0.0207
pop_best  0.6915  0.0155
non_pers  0.6935  0.0200
pers      0.7552  0.0131

pers vs non_pers (cold-start fix): delta=+0.0617  t=5.153  p=0.0009  Cohen d=3.259


In [14]:
def run_single_seed_both_coldstart(seed, n_users, n_sessions, offset_std, alpha=0.5):
    random.seed(seed); np.random.seed(seed)
    user_offsets = make_user_offsets(n_users, offset_std, seed=seed)
    pop_best_action = {
        e: EMOTION_GAMES[e][int(np.argmax([REWARD_TABLE[e][g] for g in EMOTION_GAMES[e]]))]
        for e in EMOTIONS
    }
    non_pers_bandit = LinUCBBandit(alpha=alpha)  # cold start too
    pers_bandits = {uid: LinUCBBandit(alpha=alpha) for uid in range(n_users)}

    rewards = {'random': [], 'pop_best': [], 'non_pers': [], 'pers': []}
    for uid in range(n_users):
        offset = user_offsets[uid]
        for s in range(n_sessions):
            emotion = random.choice(EMOTIONS)
            low, high = STRESS_RANGES[emotion]
            stress = random.uniform(low, high)
            x = make_vector(emotion, stress)
            valid = EMOTION_GAMES[emotion]

            a = random.choice(valid)
            rewards['random'].append(simulate_reward(emotion, a, offset))
            a = pop_best_action[emotion]
            rewards['pop_best'].append(simulate_reward(emotion, a, offset))

            a = non_pers_bandit.select(x, valid)
            r = simulate_reward(emotion, a, offset)
            non_pers_bandit.update(a, x, r)
            rewards['non_pers'].append(r)

            a = pers_bandits[uid].select(x, valid)
            r = simulate_reward(emotion, a, offset)
            pers_bandits[uid].update(a, x, r)
            rewards['pers'].append(r)
    return {k: float(np.mean(v)) for k, v in rewards.items()}


seed_results_both_cold = {k: [] for k in ['non_pers', 'pers']}
for sd in SEEDS:
    r = run_single_seed_both_coldstart(sd, n_users=N_USERS, n_sessions=N_SESSIONS, offset_std=OFFSET_STD_MAIN)
    seed_results_both_cold['non_pers'].append(r['non_pers'])
    seed_results_both_cold['pers'].append(r['pers'])

a, b = np.array(seed_results_both_cold['pers']), np.array(seed_results_both_cold['non_pers'])
t, p = stats.ttest_ind(a, b)
print(f"BOTH cold-start | non_pers={b.mean():.4f}  pers={a.mean():.4f}  "
      f"delta={a.mean()-b.mean():+.4f}  t={t:.3f}  p={p:.4f}")
print("Effect survives even when non_pers also loses its warm-start advantage")
print("-> confirms this is genuine personalisation, not an artifact of removing the prior")

BOTH cold-start | non_pers=0.6529  pers=0.7517  delta=+0.0988  t=6.478  p=0.0002
Effect survives even when non_pers also loses its warm-start advantage
-> confirms this is genuine personalisation, not an artifact of removing the prior


CRN reward-функция 


In [15]:
def reward_from_noise(emotion, game_idx, user_offset, noise_z, click_u,
                       playtime_engaged, playtime_bounce, noise_std=0.08):
    base = REWARD_TABLE[emotion][game_idx]
    if user_offset is not None:
        base = base + user_offset[game_idx]
    engagement = np.clip(base + noise_z * noise_std, 0, 1)
    playtime = playtime_engaged if click_u < engagement else playtime_bounce
    if playtime < 0.5:
        return -0.7
    return round(0.3 + 0.7 * min(1.0, playtime / 10), 3)

In [16]:
def pretrain_population_bandit_crn(n_episodes, seed, alpha=0.5):
    rng = np.random.RandomState(seed)
    bandit = LinUCBBandit(alpha=alpha)
    for _ in range(n_episodes):
        emotion = rng.choice(EMOTIONS)
        low, high = STRESS_RANGES[emotion]
        stress = rng.uniform(low, high)
        x = make_vector(emotion, stress)
        valid = EMOTION_GAMES[emotion]
        a = rng.choice(valid)
        z, u, pe, pb = rng.normal(), rng.uniform(), rng.uniform(5, 15), rng.uniform(0, 1)
        r = reward_from_noise(emotion, a, None, z, u, pe, pb)
        bandit.update(a, x, r)
    return bandit

In [17]:
def run_single_seed_crn(seed, n_users, n_sessions, offset_std, warm_start_pers=False,
                         alpha_pers=0.5, alpha_nonpers=0.5, pretrain_episodes=5000):
    master_rng = np.random.RandomState(seed)
    user_offsets = make_user_offsets(n_users, offset_std, seed=seed)
    pop_best_action = {
        e: EMOTION_GAMES[e][int(np.argmax([REWARD_TABLE[e][g] for g in EMOTION_GAMES[e]]))]
        for e in EMOTIONS
    }

    pretrained = pretrain_population_bandit_crn(pretrain_episodes, seed=seed, alpha=alpha_nonpers)
    non_pers_bandit = clone_bandit(pretrained)  # warm-started (как раньше)
    pers_bandits = {
        uid: (clone_bandit(pretrained) if warm_start_pers else LinUCBBandit(alpha=alpha_pers))
        for uid in range(n_users)
    }  # cold-start по умолчанию

    per_user_rewards = {k: {uid: [] for uid in range(n_users)}
                         for k in ['random', 'pop_best', 'non_pers', 'pers']}

    for uid in range(n_users):
        offset = user_offsets[uid]
        for s in range(n_sessions):
            emotion = master_rng.choice(EMOTIONS)
            low, high = STRESS_RANGES[emotion]
            stress = master_rng.uniform(low, high)
            x = make_vector(emotion, stress)
            valid = EMOTION_GAMES[emotion]

            # CRN: заранее генерируем шум на КАЖДУЮ arm в раунде
            noise = {}
            for arm in valid:
                noise[arm] = dict(
                    z=master_rng.normal(), u=master_rng.uniform(),
                    pe=master_rng.uniform(5, 15), pb=master_rng.uniform(0, 1)
                )

            def reward_for(arm):
                n = noise[arm]
                return reward_from_noise(emotion, arm, offset, n['z'], n['u'], n['pe'], n['pb'])

            a = master_rng.choice(valid)
            per_user_rewards['random'][uid].append(reward_for(a))

            a = pop_best_action[emotion]
            per_user_rewards['pop_best'][uid].append(reward_for(a))

            a = non_pers_bandit.select(x, valid)
            r = reward_for(a); non_pers_bandit.update(a, x, r)
            per_user_rewards['non_pers'][uid].append(r)

            a = pers_bandits[uid].select(x, valid)
            r = reward_for(a); pers_bandits[uid].update(a, x, r)
            per_user_rewards['pers'][uid].append(r)

    return per_user_rewards

In [18]:
all_seed_peruser = {k: [] for k in ['random', 'pop_best', 'non_pers', 'pers']}   # pooled: n=150
seed_level_means_crn = {k: [] for k in ['random', 'pop_best', 'non_pers', 'pers']}  # n=5

for sd in SEEDS:
    pur = run_single_seed_crn(sd, n_users=N_USERS, n_sessions=N_SESSIONS,
                               offset_std=OFFSET_STD_MAIN, warm_start_pers=False)
    for k in all_seed_peruser:
        user_means = [np.mean(pur[k][uid]) for uid in range(N_USERS)]
        all_seed_peruser[k].extend(user_means)
        seed_level_means_crn[k].append(np.mean(user_means))

print("① Seed-level (n=5):")
for k in seed_level_means_crn:
    v = np.array(seed_level_means_crn[k])
    print(f"  {k:10s} mean={v.mean():.4f}  std={v.std():.4f}")
t, p = stats.ttest_ind(seed_level_means_crn['pers'], seed_level_means_crn['non_pers'])
print(f"  pers vs non_pers: t={t:.3f}  p={p:.4f}")

print("\n② Pooled across seeds (n=150 независимых юзеров):")
for k in all_seed_peruser:
    v = np.array(all_seed_peruser[k])
    print(f"  {k:10s} mean={v.mean():.4f}  std={v.std():.4f}  n={len(v)}")

a, b = np.array(all_seed_peruser['pers']), np.array(all_seed_peruser['non_pers'])
t, p = stats.ttest_ind(a, b)
d = (a.mean() - b.mean()) / np.sqrt((a.std(ddof=1)**2 + b.std(ddof=1)**2) / 2)
se = np.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))
ci_low, ci_high = stats.t.interval(0.95, len(a)+len(b)-2, loc=a.mean()-b.mean(), scale=se)
print(f"\n  delta={a.mean()-b.mean():+.4f}  t={t:.3f}  p={p:.2e}  d={d:.3f}  95%CI=[{ci_low:.4f},{ci_high:.4f}]")

t_paired, p_paired = stats.ttest_rel(a, b)
print(f"  paired t-test (валиден благодаря CRN): t={t_paired:.3f}  p={p_paired:.2e}")

① Seed-level (n=5):
  random     mean=0.6221  std=0.0162
  pop_best   mean=0.6892  std=0.0159
  non_pers   mean=0.6876  std=0.0132
  pers       mean=0.7538  std=0.0176
  pers vs non_pers: t=6.016  p=0.0003

② Pooled across seeds (n=150 независимых юзеров):
  random     mean=0.6221  std=0.0807  n=150
  pop_best   mean=0.6892  std=0.1175  n=150
  non_pers   mean=0.6876  std=0.1136  n=150
  pers       mean=0.7538  std=0.0724  n=150

  delta=+0.0662  t=6.003  p=5.62e-09  d=0.693  95%CI=[0.0445,0.0879]
  paired t-test (валиден благодаря CRN): t=8.181  p=1.15e-13
